In [1]:
import random as rnd

class Taximap():

    map = []
    destinations = {}

    def generate_map(map_x_size, map_y_size, number_of_destinations):
        Taximap.map = []
        Taximap.destinations = {}

        ### Blank Map
        for _ in range(map_y_size):
            Taximap.map.append([])
            for _ in range(map_x_size):
                Taximap.map[-1].append(": ")

        ### Destinations
        total_spaces = ((map_y_size-2) + map_x_size)*2 ### Only the border tiles
        destinations = rnd.sample(range(0,total_spaces), number_of_destinations)
        for dest_num, location in enumerate(destinations):
            if location < map_x_size: ### Top Row
                locy = 0
                locx = location
            elif total_spaces - map_x_size < location: ### Bottom Row
                locy = map_y_size-1
                locx = location - (total_spaces-map_x_size)
            else:   # The Rest
                locy = ((location - map_x_size)//2) + 1
                locx = (location - map_x_size)%2 * (map_x_size-1)

            Taximap.map[locy][locx] = str(dest_num+1) + " "
            Taximap.destinations[(locx, locy)] = dest_num
        
        # total_spaces = map_y_size*map_x_size
        # destinations = rnd.sample(range(0,total_spaces), self.destination_amount)
        # for dest_num, location in enumerate(destinations):
        #     locx, locy = location//6, location%6 ### X and Y coordinates of the destination
        #     self.map[locy][locx] = str(dest_num+1) + " "

    def render_blank_map():
        map = []
        # [print(x) for x in Taximap.map]
        map.append("x"+("-"*((len(Taximap.map[0])*2)+1))+"x") ### Top Wall
        for row in Taximap.map:
            map.append("| " + ''.join(row) + "|")
        map.append("x"+("-"*((len(Taximap.map[0])*2)+1))+"x") ### Bottom Wall

        for line in map:
            print(line)

        return "\n".join(map)

In [2]:
import numpy as np

class Taxi:

    posx, posy = -1, -1 ### 36
    action = -1 ### 0: Go left, 1: Go right, 2: Go up, 3: Go down + 4: Pick up + 5: Drop off
    passenger_state = -1 ### 0-4: Destinations + 5: Inside the Taxi
    destination_position = -1 ### 5
    reward = 0

    # Hyperparameters
    learning_rate = 0.1
    discount_factor = 0.95
    epsilon = 0.95

    num_of_states = 0
    num_of_actions = 6
    q_table = np.zeros([num_of_states, num_of_actions])
    
    def initialize_environment(map_size_x, map_size_y, destination_amount, reset_q_table = False):
        Taximap.generate_map(map_size_x, map_size_y, destination_amount)

        if reset_q_table:
            Taxi.num_of_states = map_size_x*map_size_y*(destination_amount+1)*destination_amount ### Taxi positions * passenger states * destination positions
            Taxi.q_table = np.zeros([Taxi.num_of_states, Taxi.num_of_actions])
            print(Taxi.num_of_states)

    def random_start():
        Taxi.posx, Taxi.posy = rnd.randrange(len(Taximap.map[0])), rnd.randrange(len(Taximap.map))
        Taxi.passenger_state, Taxi.destination_position = rnd.sample(range(0, len(Taximap.destinations)), 2)
        Taxi.reward = 0

    def get_current_state():
        state = ((Taxi.posy * len(Taximap.map[0]) + Taxi.posx) * (len(Taximap.destinations) + 1) + Taxi.passenger_state) * len(Taximap.destinations) + Taxi.destination_position

        return state
    
    def update_q_table(old_state, old_value, next_value, reward):
        Taxi.q_table[old_state, Taxi.action] = (1 - Taxi.learning_rate) * old_value + Taxi.learning_rate * (reward + Taxi.discount_factor * next_value)

    def step():
        old_state = Taxi.get_current_state()

        if rnd.uniform(0, 1) < Taxi.epsilon:
            Taxi.action = rnd.randrange(6)
        else:
            Taxi.action = np.argmax(Taxi.q_table[old_state])

        reward = Taxi.act()
        Taxi.reward += reward

        terminated = (reward == 20)

        if terminated:
            next_value = 0
        else:
            next_value = np.max(Taxi.q_table[Taxi.get_current_state()])

        Taxi.update_q_table(
            old_state=old_state,
            old_value=Taxi.q_table[old_state, Taxi.action],
            next_value=next_value,
            reward=reward
        )

        return terminated
    

    def step():
        old_state = Taxi.get_current_state()
        if rnd.uniform(0, 1) < Taxi.epsilon:
            Taxi.action = rnd.randrange(6) # Explore
        else:
            Taxi.action = np.argmax(Taxi.q_table[old_state]) # Exploit

        reward = Taxi.act()

        Taxi.reward += reward

        terminated = (reward == 20)

        if terminated:
            next_value =  0
        else:
            next_value = np.max(Taxi.q_table[Taxi.get_current_state()])
        
        Taxi.update_q_table(
                            old_state = old_state,
                            old_value = Taxi.q_table[old_state, Taxi.action], 
                            next_value = next_value,
                            reward = reward
                            )
        
        return terminated
        

    def act():
        actions = [Taxi.move_left, Taxi.move_right, Taxi.move_up, Taxi.move_down, Taxi.pick_up, Taxi.drop_off]
        return actions[Taxi.action]()

    def move_left():
        if Taxi.posx > 0:
            Taxi.posx -= 1

        return -1

    def move_right():
        if Taxi.posx < len(Taximap.map[0])-1:
            Taxi.posx += 1

        return -1

    def move_up():
        if Taxi.posy > 0:
            Taxi.posy -= 1

        return -1

    def move_down():
        if Taxi.posy < len(Taximap.map)-1:
            Taxi.posy += 1

        return -1
        
    def pick_up():
        if Taxi.passenger_state < len(Taximap.destinations) and (Taxi.posx, Taxi.posy) in Taximap.destinations and Taximap.destinations[(Taxi.posx, Taxi.posy)] == Taxi.passenger_state:
            # Taximap.map[Taxi.posy][Taxi.posx] = f"{Taximap.destinations[(Taxi.posx, Taxi.posy)]+1} " ### Remove the passenger from the map
            Taxi.passenger_state = len(Taximap.destinations) ### Place the passenger inside of the taxi
            return -1 ### Passenger picked up
        else:
            return -10 ### Illegal "pick up"

    def drop_off():
        if Taxi.passenger_state == len(Taximap.destinations) and (Taxi.posx, Taxi.posy) in Taximap.destinations:
            # Taximap.map[Taxi.posy][Taxi.posx] = f"{Taximap.map[Taxi.posy][Taxi.posx][0]}P" ### Add the passenger to the map     
            Taxi.passenger_state = Taximap.destinations[(Taxi.posx, Taxi.posy)] ### Place the passenger to the current position       
            if Taxi.passenger_state == Taxi.destination_position:
                return 20 ### Correct destination
            else:
                return -1 ### Incorrect destination
        else:
            return -10 ### Illegal "drop off"

    def render_map():
        map = []
        # [print(x) for x in Taximap.map]
        map.append("x"+("-"*((len(Taximap.map[0])*2)+1))+"x") ### Top Wall
        
        for rn, row in enumerate(Taximap.map):
            render = []
            for cn, col in enumerate(row):
                plot = ":"
                current_pos = (cn, rn)

                if current_pos == (Taxi.posx, Taxi.posy):
                    if Taxi.passenger_state == len(Taximap.destinations): ### If the passenger is inside the taxi
                        plot = "T"
                    else:
                        plot = "t"
                elif current_pos in Taximap.destinations:
                    plot = str(Taximap.destinations[current_pos])

                if ( ### Passanger on a destination
                    Taxi.passenger_state < len(Taximap.destinations)
                    and current_pos in Taximap.destinations
                    and Taximap.destinations[current_pos] == Taxi.passenger_state
                ):
                    plot += "p"

                elif ( ### Destinations
                    current_pos in Taximap.destinations
                    and Taximap.destinations[current_pos] == Taxi.destination_position
                ):
                    plot += "d"
                else:
                    plot += " "
                    
                render.append(plot)

            map.append("| " + ''.join(render) + "|")

        map.append("x"+("-"*((len(Taximap.map[0])*2)+1))+"x") ### Bottom Wall

        for line in map:
            print(line)

        return "\n".join(map)

In [3]:
def train_taxi():
    epoch_count = 10000
    max_step_count = 150
    rewards = []

    Taxi.initialize_environment(6, 6, 5, reset_q_table=True) ### Set the map
    print("Training Started!")
    for n in range(epoch_count):
        terminated = False
        Taxi.random_start()

        for _ in range(max_step_count):
            terminated = Taxi.step()

            if terminated: ### Terminated
                break
        else:
            ... ### Truncated

        rewards.append(Taxi.reward)
        Taxi.epsilon = max(0.01, Taxi.epsilon * 0.9995) ### Epsilon decay

        if n==0 or (n+1) % 1000 == 0: ### Report per 1000
            print(f"Epoch {n+1} - Average Reward: {np.mean(rewards[-1000:]):.3f}, Epsilon value: {Taxi.epsilon:.3f}")

def test_taxi():

    Taxi.random_start()
    Taximap.render_blank_map()
    Taxi.epsilon = 0 ### Using only exploited knowledge
    for _ in range(100):
        terminated = Taxi.step()
        # [print() for _ in range(25)]
        Taxi.render_map()
        input()
        if terminated:
            print("Testing successful!")
            break
    else:
        print("Testing failed")
        input()

In [4]:
train_taxi()

1080
Training Started!
Epoch 1 - Average Reward: -591.000, Epsilon value: 0.950
Epoch 1000 - Average Reward: -397.839, Epsilon value: 0.576
Epoch 2000 - Average Reward: -107.349, Epsilon value: 0.349
Epoch 3000 - Average Reward: -18.546, Epsilon value: 0.212
Epoch 4000 - Average Reward: -2.467, Epsilon value: 0.129
Epoch 5000 - Average Reward: 3.851, Epsilon value: 0.078
Epoch 6000 - Average Reward: 6.189, Epsilon value: 0.047
Epoch 7000 - Average Reward: 7.446, Epsilon value: 0.029
Epoch 8000 - Average Reward: 8.364, Epsilon value: 0.017
Epoch 9000 - Average Reward: 8.980, Epsilon value: 0.011
Epoch 10000 - Average Reward: 9.141, Epsilon value: 0.010


In [5]:
test_taxi()

x-------------x
| : : : : 2 : |
| : : : : : : |
| 4 : : : : : |
| : : : : : : |
| : : : : : : |
| : 3 5 : : 1 |
x-------------x
x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : t : |
| : : : : : : |
| : : : : : : |
| : 2 4p: : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : t : : |
| : : : : : : |
| : : : : : : |
| : 2 4p: : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : : : |
| : : : t : : |
| : : : : : : |
| : 2 4p: : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : : : |
| : : : : : : |
| : : : t : : |
| : 2 4p: : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : : : |
| : : : : : : |
| : : t : : : |
| : 2 4p: : 0 |
x-------------x


...

x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : : : |
| : : : : : : |
| T : : : : : |
| : 2 4 : : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| 3d: : : : : |
| T : : : : : |
| : : : : : : |
| : 2 4 : : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| Td: : : : : |
| : : : : : : |
| : : : : : : |
| : 2 4 : : 0 |
x-------------x


x-------------x
| : : : : 1 : |
| : : : : : : |
| tp: : : : : |
| : : : : : : |
| : : : : : : |
| : 2 4 : : 0 |
x-------------x


Testing successful!
